In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, roc_auc_score)
import time


In [3]:
!git clone https://github.com/ZiadXI/Brazilian-E-Commerce.git

fatal: destination path 'Brazilian-E-Commerce' already exists and is not an empty directory.


In [4]:
# 1. Load the cleaned dataset
df = pd.read_csv('Brazilian-E-Commerce/data/clean_orders/orders.csv')
# 2. Convert dates to datetime to calculate 'expected_days'
df['order_date'] = pd.to_datetime(df['order_date'])
df['expected_delivery_date'] = pd.to_datetime(df['expected_delivery_date'])
df['expected_days'] = (df['expected_delivery_date'] - df['order_date']).dt.days

# 3. NO LEAKAGE RULE: Select ONLY the allowed features + target
# We specifically do NOT include actual_delivery_days, delivery_status, delivery_delay, etc.
allowed_columns = [
    'courier', 'expected_days', 'weather', 'season',
    'area', 'category', 'order_month', 'order_hour',
    'is_late' # Target
]

# Create our strict modeling dataframe and drop any rows with NaN in these specific columns
model_df = df[allowed_columns].dropna().copy()

print(f"Dataset shape ready for ML: {model_df.shape}")


Dataset shape ready for ML: (113314, 9)


In [5]:
# 1. Define X (Features) and y (Target)
X = model_df.drop(columns=['is_late'])
y = model_df['is_late']

# 2. Train/Test Split (80/20) - Stratified ensures same % of late orders in both sets!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Identify categorical vs numerical columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# 4. Preprocessing: Scale numbers, One-Hot Encode categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Fit on training data, transform both
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Get the new column names after OneHotEncoding
encoded_cat_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(encoded_cat_cols)

print(f"Number of features after encoding: {X_train_encoded.shape[1]}")


Number of features after encoding: 114


In [6]:
print("Train shape:", X_train_encoded.shape)
print("Test shape:", X_test_encoded.shape)
print("Target distribution (train):")
print(y_train.value_counts(normalize=True))

Train shape: (90651, 114)
Test shape: (22663, 114)
Target distribution (train):
is_late
0    0.935632
1    0.064368
Name: proportion, dtype: float64


# Random Forest - HyperParameter Tuning

In [7]:
print("Tuning Random Forest...")
t0 = time.time()

rf_param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_dist, n_iter=10, cv=3, scoring='f1',
    random_state=42, n_jobs=-1
)
rf_search.fit(X_train_encoded, y_train)
rf_best = rf_search.best_estimator_

print("RF best params:", rf_search.best_params_)
print(f"Tuning time: {time.time()-t0:.1f}s")

Tuning Random Forest...
RF best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 15, 'class_weight': 'balanced'}
Tuning time: 778.9s


# Random Forset Evaluation Metrics

In [9]:
rf_pred = rf_best.predict(X_test_encoded)
rf_proba = rf_best.predict_proba(X_test_encoded)[:, 1]

rf_results = {
    'Accuracy':  accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred),
    'Recall':    recall_score(y_test, rf_pred),
    'F1':        f1_score(y_test, rf_pred),
    'ROC-AUC':   roc_auc_score(y_test, rf_proba),
}
print("Random Forest Results :")
for k, v in rf_results.items():
    print(f"{k}: {v:.4f}")

Random Forest Results :
Accuracy: 0.8226
Precision: 0.1810
Recall: 0.4983
F1: 0.2656
ROC-AUC: 0.7533


he Random Forest model detects approximately half of the actual late deliveries (Recall = 49.83%), meaning it can identify around 50% of orders that will actually be late. However, its low Precision (18.10%) indicates a high number of false alarms: among orders predicted as late, only about 18% are actually late. This represents a trade-off where the model prioritizes detecting more late deliveries, which may be related to using class_weight='balanced'. However, the impact of balancing should be confirmed by comparing it with a model using class_weight=None.

In [12]:
rf_best = rf_search.best_estimator_
import joblib
joblib.dump(rf_best, 'rf_best_model.pkl')

['rf_best_model.pkl']

# Random Forest Balanced VS. No Weight

In [10]:
params_no_weight = {
    k: v for k, v in rf_search.best_params_.items()
    if k != 'class_weight'
}

params_no_weight['class_weight'] = None

rf_no_weight = RandomForestClassifier(
    **params_no_weight,
    random_state=42,
    n_jobs=-1
)

rf_no_weight.fit(X_train_encoded, y_train)

pred_nw = rf_no_weight.predict(X_test_encoded)
proba_nw = rf_no_weight.predict_proba(X_test_encoded)[:, 1]

rf_no_weight_results = {
    'Accuracy': accuracy_score(y_test, pred_nw),
    'Precision': precision_score(y_test, pred_nw, zero_division=0),
    'Recall': recall_score(y_test, pred_nw, zero_division=0),
    'F1': f1_score(y_test, pred_nw, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, proba_nw)
}

print("\n Random Forest (Without class_weight) - Comparison :")

for k, v in rf_no_weight_results.items():
    print(f"{k}: {v:.4f}")


 Random Forest (Without class_weight) - Comparison :
Accuracy: 0.9356
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
ROC-AUC: 0.7526


Without class weighting, the Random Forest achieved a higher Accuracy (93.56%), but completely failed to identify late deliveries (Recall = 0%, F1 = 0%). In contrast, using class_weight='balanced' reduced Accuracy to 82.26% but substantially improved the detection of late deliveries, achieving 49.83% Recall and 26.56% F1. Therefore, class_weight='balanced' is more appropriate for this imbalanced classification problem because the main goal is to detect late deliveries rather than simply maximize overall Accuracy.

# Threshold Tuning - Optimizing the Decision Boundary

In [36]:
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

for t in thresholds:
    pred_t = (rf_proba >= t).astype(int)
    print(f"Threshold={t}: "
          f"Precision={precision_score(y_test, pred_t):.3f}, "
          f"Recall={recall_score(y_test, pred_t):.3f}, "
          f"F1={f1_score(y_test, pred_t):.3f}")

Threshold=0.3: Precision=0.071, Recall=0.982, F1=0.133
Threshold=0.35: Precision=0.087, Recall=0.906, F1=0.159
Threshold=0.4: Precision=0.111, Recall=0.772, F1=0.194
Threshold=0.45: Precision=0.140, Recall=0.657, F1=0.231
Threshold=0.5: Precision=0.181, Recall=0.498, F1=0.266


In [37]:
thresholds = [0.5, 0.55, 0.6, 0.65, 0.7]

for t in thresholds:
    pred_t = (rf_proba >= t).astype(int)
    print(f"Threshold={t}: "
          f"Precision={precision_score(y_test, pred_t):.3f}, "
          f"Recall={recall_score(y_test, pred_t):.3f}, "
          f"F1={f1_score(y_test, pred_t):.3f}")

Threshold=0.5: Precision=0.181, Recall=0.498, F1=0.266
Threshold=0.55: Precision=0.227, Recall=0.365, F1=0.280
Threshold=0.6: Precision=0.262, Recall=0.281, F1=0.271
Threshold=0.65: Precision=0.323, Recall=0.172, F1=0.225
Threshold=0.7: Precision=0.415, Recall=0.082, F1=0.136


In [39]:
best_threshold = 0.55
rf_pred_final = (rf_proba >= best_threshold).astype(int)

rf_results_final = {
    'Accuracy':  accuracy_score(y_test, rf_pred_final),
    'Precision': precision_score(y_test, rf_pred_final),
    'Recall':    recall_score(y_test, rf_pred_final),
    'F1':        f1_score(y_test, rf_pred_final),
    'ROC-AUC':   roc_auc_score(y_test, rf_proba),
}

print(f"Random Forest Results (threshold={best_threshold}):")
for k, v in rf_results_final.items():
    print(f"{k}: {v:.4f}")

Random Forest Results (threshold=0.55):
Accuracy: 0.8791
Precision: 0.2268
Recall: 0.3646
F1: 0.2796
ROC-AUC: 0.7533


**Instead of using the default 0.5 threshold, we tested multiple thresholds
(0.3 to 0.7) to find the optimal decision boundary for classifying orders
as "late." The best F1-score (0.280) was achieved at threshold=0.55,
compared to 0.266 at the default 0.5 — a 5.3% improvement.**

**This adjustment increased Accuracy from 82.3% to 87.9% and Precision from
18.1% to 22.7%, meaning the model produces fewer false alarms and is more
reliable when it predicts a delay. However, Recall dropped from 49.8% to
36.5%, so the model now catches fewer of the actual late deliveries.**

**This is a business trade-off, not a purely technical one: threshold=0.55
is preferable if the priority is reducing wasted resources on false
alarms, while the default 0.5 remains better if catching as many late
deliveries as possible is the priority (e.g., for proactive customer
communication). We recommend threshold=0.55 for operational efficiency,
but suggest revisiting this choice based on the actual cost of a missed
late delivery versus a false alarm.**

In [43]:
for w in [3, 5, 8, 10]:
    rf_w = RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=2, min_samples_leaf=1,
        class_weight={0:1, 1:w}, random_state=42, n_jobs=-1
    )
    rf_w.fit(X_train_encoded, y_train)
    proba_w = rf_w.predict_proba(X_test_encoded)[:,1]
    pred_w = (proba_w >= 0.5).astype(int)
    print(f"weight=1:{w} F1={f1_score(y_test, pred_w):.3f}")

weight=1:3 F1=0.001
weight=1:5 F1=0.104
weight=1:8 F1=0.260
weight=1:10 F1=0.279


**We experimented with different class_weight ratios to give the model more focus on late deliveries, ranging from a low ratio to a high ratio. The results showed that the automatic 'balanced' setting used originally was actually the best option, and manually changing it did not improve performance.**

# KNN -Hyper Parameter Tuning

In [31]:
from sklearn.model_selection import train_test_split as tts

X_knn_tune, _, y_knn_tune, _ = tts(
    X_train_encoded,
    y_train,
    train_size=20000,
    random_state=42,
    stratify=y_train
)

print("Tuning KNN...")

t0 = time.time()

knn_param_dist = {
    'n_neighbors': [5, 9, 15, 21, 31],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_search = RandomizedSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    knn_param_dist,
    n_iter=8,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

knn_search.fit(X_knn_tune, y_knn_tune)

print("KNN best params:", knn_search.best_params_)
print(f"Tuning time: {time.time() - t0:.1f}s")

Tuning KNN...
KNN best params: {'weights': 'distance', 'n_neighbors': 5, 'metric': 'manhattan'}
Tuning time: 168.5s


# KNN - Final Training

In [32]:
knn_best = KNeighborsClassifier(
    **knn_search.best_params_,
    n_jobs=-1
)

knn_best.fit(X_train_encoded, y_train)

print("Final KNN trained on the full training set.")

Final KNN trained on the full training set.


# KNN - Model Evaluation

In [33]:
knn_pred = knn_best.predict(X_test_encoded)
knn_proba = knn_best.predict_proba(X_test_encoded)[:, 1]

knn_results = {
    'Accuracy': accuracy_score(y_test, knn_pred),
    'Precision': precision_score(y_test, knn_pred, zero_division=0),
    'Recall': recall_score(y_test, knn_pred, zero_division=0),
    'F1': f1_score(y_test, knn_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, knn_proba)
}

print("KNN Results:")

for k, v in knn_results.items():
    print(f"{k}: {v:.4f}")

KNN Results:
Accuracy: 0.9291
Precision: 0.2439
Recall: 0.0480
F1: 0.0802
ROC-AUC: 0.6012


In [34]:
joblib.dump(knn_best, "knn_baseline.pkl")

['knn_baseline.pkl']

# KNN - Dimensionality Reduction with SVD

In [27]:
from sklearn.decomposition import TruncatedSVD

print("Applying SVD...")

svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

X_train_reduced = svd.fit_transform(X_train_encoded)

print("Original shape:", X_train_encoded.shape)
print("Reduced shape:", X_train_reduced.shape)
print(f"Explained variance ratio (sum): {svd.explained_variance_ratio_.sum():.4f}")

Applying SVD...
Original shape: (90651, 114)
Reduced shape: (90651, 100)
Explained variance ratio (sum): 0.9997


# KNN - Hyperparameter Tuning with SVD

In [28]:
from sklearn.model_selection import train_test_split as tts

X_knn_tune, _, y_knn_tune, _ = tts(
    X_train_reduced,
    y_train,
    train_size=20000,
    random_state=42,
    stratify=y_train
)

print("Tuning KNN with SVD...")

t0 = time.time()

knn_param_dist = {
    'n_neighbors': [5, 9, 15, 21, 31],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_search_svd = RandomizedSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    knn_param_dist,
    n_iter=8,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

knn_search_svd.fit(X_knn_tune, y_knn_tune)

print("KNN best params:", knn_search_svd.best_params_)
print(f"Tuning time: {time.time() - t0:.1f}s")

Tuning KNN with SVD...
KNN best params: {'weights': 'distance', 'n_neighbors': 5, 'metric': 'euclidean'}
Tuning time: 148.2s


# KNN + SVD - Final Training

In [29]:
knn_svd = KNeighborsClassifier(
    **knn_search_svd.best_params_,
    n_jobs=-1
)

knn_svd.fit(X_train_reduced, y_train)

print("Final KNN with SVD trained.")

Final KNN with SVD trained.


**The baseline KNN model achieved a high Accuracy of 92.91%, but this result is misleading due to the severe class imbalance in the dataset. The model detected only 4.8% of the actual late deliveries (Recall), meaning it missed approximately 95% of late orders. Although the Precision reached 24.39%, the low Recall and F1-score of 8.02% indicate that the model is not effective for identifying late deliveries. This suggests that the class imbalance significantly limits the baseline KNN model's ability to recognize the minority Late class.**

# KNN + SVD - Model Evaluation

In [30]:
X_test_reduced = svd.transform(X_test_encoded)

knn_svd_pred = knn_svd.predict(X_test_reduced)
knn_svd_proba = knn_svd.predict_proba(X_test_reduced)[:, 1]

knn_svd_results = {
    'Accuracy': accuracy_score(y_test, knn_svd_pred),
    'Precision': precision_score(y_test, knn_svd_pred, zero_division=0),
    'Recall': recall_score(y_test, knn_svd_pred, zero_division=0),
    'F1': f1_score(y_test, knn_svd_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, knn_svd_proba)
}

print("\nKNN + SVD Results:")

for k, v in knn_svd_results.items():
    print(f"{k}: {v:.4f}")


KNN + SVD Results:
Accuracy: 0.9306
Precision: 0.2767
Recall: 0.0480
F1: 0.0818
ROC-AUC: 0.5984


**SVD did not improve KNN performance. Although dimensionality reduction slightly increased Accuracy, Recall, F1-score, and ROC-AUC all decreased. Therefore, SVD was not beneficial for this KNN task.**

In [35]:
joblib.dump(knn_svd, "knn_svd_model.pkl")
joblib.dump(svd, "knn_svd_transformer.pkl")
joblib.dump(knn_svd_results, "knn_svd_results.pkl")

['knn_svd_results.pkl']

# **Applying Smote**

## KNN - Handling Class Imbalance with SMOTE

In [20]:
from imblearn.over_sampling import SMOTE

In [21]:
print("Applying SMOTE...")

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_encoded,
    y_train
)

print("Original training shape:", X_train_encoded.shape)
print("Balanced training shape:", X_train_smote.shape)

print("\nClass distribution before SMOTE:")
print(y_train.value_counts())

print("\nClass distribution after SMOTE:")
print(y_train_smote.value_counts())

Applying SMOTE...
Original training shape: (90651, 114)
Balanced training shape: (169632, 114)

Class distribution before SMOTE:
is_late
0    84816
1     5835
Name: count, dtype: int64

Class distribution after SMOTE:
is_late
0    84816
1    84816
Name: count, dtype: int64


## KNN - Hyperparameter Tuning with SMOTE

In [22]:
from sklearn.model_selection import train_test_split as tts

X_knn_tune, _, y_knn_tune, _ = tts(
    X_train_smote,
    y_train_smote,
    train_size=20000,
    random_state=42,
    stratify=y_train_smote
)

print("Tuning KNN with SMOTE...")

t0 = time.time()

knn_param_dist = {
    'n_neighbors': [3, 5, 7, 9, 15, 21, 31],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_search_smote = RandomizedSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    knn_param_dist,
    n_iter=10,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

knn_search_smote.fit(X_knn_tune, y_knn_tune)

print("KNN best params:", knn_search_smote.best_params_)
print(f"Tuning time: {time.time() - t0:.1f}s")

Tuning KNN with SMOTE...
KNN best params: {'weights': 'distance', 'n_neighbors': 5, 'metric': 'manhattan'}
Tuning time: 159.5s


## KNN + SMOTE - Final Training

In [25]:
knn_smote = KNeighborsClassifier(
    **knn_search_smote.best_params_,
    n_jobs=-1
)

knn_smote.fit(X_train_smote, y_train_smote)

print("Final KNN with SMOTE trained.")

Final KNN with SMOTE trained.


## KNN + SMOTE - Model Evaluation

In [24]:
knn_smote_pred = knn_smote.predict(X_test_encoded)
knn_smote_proba = knn_smote.predict_proba(X_test_encoded)[:, 1]

knn_smote_results = {
    'Accuracy': accuracy_score(y_test, knn_smote_pred),
    'Precision': precision_score(y_test, knn_smote_pred, zero_division=0),
    'Recall': recall_score(y_test, knn_smote_pred, zero_division=0),
    'F1': f1_score(y_test, knn_smote_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, knn_smote_proba)
}

print("\nKNN + SMOTE Results:")

for k, v in knn_smote_results.items():
    print(f"{k}: {v:.4f}")


KNN + SMOTE Results:
Accuracy: 0.8368
Precision: 0.1255
Recall: 0.2570
F1: 0.1686
ROC-AUC: 0.6064


**Insight: SMOTE improved KNN's ability to detect late deliveries, but the model still produced many false alarms and had weak overall discrimination. Random Forest with balanced class weights performed substantially better and is therefore more suitable for this task.**

In [26]:
joblib.dump(knn_smote_results, 'knn_smote_results.pkl')

['knn_smote_results.pkl']

| Model           |   Accuracy |  Precision |   Recall  |       F1  |    ROC-AUC |
| --------------- | ---------: | ---------: | ---------: | ---------: | ---------: |
| KNN Baseline    |     92.91% |     24.39% |      4.80% |      8.02% |     60.12% |
| KNN + SVD       | **93.06%** | **27.67%** |      4.80% |      8.18% |     59.84% |
| **KNN + SMOTE** |     83.68% |     12.55% | **25.70%** | **16.86%** | **60.64%** |


**Insight**


**The baseline KNN model achieved high Accuracy (92.91%), but its very low Recall (4.80%) shows that it failed to identify most late deliveries. Applying SVD slightly increased Accuracy to 93.06% and Precision to 27.67%, but Recall remained at 4.80% and ROC-AUC decreased slightly, indicating that dimensionality reduction did not meaningfully improve the model's ability to detect late orders.**

**In contrast, SMOTE significantly improved Recall from 4.80% to 25.70% and increased F1-score from 8.02% to 16.86%, showing that balancing the training data helped KNN identify more late deliveries. However, this improvement came at the cost of lower Accuracy and Precision. Overall, SMOTE was the most effective modification for improving KNN's ability to detect the minority Late class, although KNN still showed limited overall performance.**